### Making a request


In [20]:
# Note that Boto3 version should be >= 1.34
# In case you face an issue with converse in bedrock runtime,
# Uninstall boto3 and install it again with pip install boto3
import boto3


client = boto3.client('bedrock-runtime')
model_id = "global.anthropic.claude-sonnet-4-20250514-v1:0"

In [10]:
user_message = {
    "role": "user",
    "content": [
        {"text": "What's 1+1?"}
    ]
}


In [11]:
response = client.converse(
    modelId=model_id,
    messages=[user_message]
)

In [12]:
response

{'ResponseMetadata': {'RequestId': '0be9f509-ae5d-4937-9b54-a2f7da3b4da0',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Wed, 08 Apr 2026 03:56:23 GMT',
   'content-type': 'application/json',
   'content-length': '322',
   'connection': 'keep-alive',
   'x-amzn-requestid': '0be9f509-ae5d-4937-9b54-a2f7da3b4da0'},
  'RetryAttempts': 0},
 'output': {'message': {'role': 'assistant',
   'content': [{'text': '1 + 1 = 2'}]}},
 'stopReason': 'end_turn',
 'usage': {'inputTokens': 14,
  'outputTokens': 13,
  'totalTokens': 27,
  'cacheReadInputTokens': 0,
  'cacheWriteInputTokens': 0},
 'metrics': {'latencyMs': 3366}}

In [13]:
response["output"]["message"]["content"][0]["text"]


'1 + 1 = 2'

## Helper Functions for Message Management


In [14]:
def add_user_message(messages, text):
    user_message = {
        "role": "user",
        "content": [
            {"text": text}
        ]
    }
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {
        "role": "assistant", 
        "content": [
            {"text": text}
        ]
    }
    messages.append(assistant_message)

def chat(messages):
    response = client.converse(
        modelId=model_id,
        messages=messages
    )
    return response["output"]["message"]["content"][0]["text"]

In [15]:
# Make a starting list of messages
messages = []

# Add in the initial user question of "What's 1+1?"
add_user_message(messages, "What's 1+1?")
print(messages)


# Pass the list of messages into chat to get an answer
answer = chat(messages)


# Take the answer and add it as an assistant message into our list
add_assistant_message(messages, answer)
print(messages)

[{'role': 'user', 'content': [{'text': "What's 1+1?"}]}]
[{'role': 'user', 'content': [{'text': "What's 1+1?"}]}, {'role': 'assistant', 'content': [{'text': '1 + 1 = 2'}]}]


In [16]:
# Add in the user's followup question
add_user_message(messages, "And 3 more added to that?")

# Call chat again with the list of messages to get a final answer
answer = chat(messages)
print(answer)

2 + 3 = 5


### Conversation

In [ ]:
messages = []

while True:
    # Get the user input
    user_input = input("> You: ")
    print(f"> You: {user_input}")

    # Add the user message to the list of messages
    add_user_message(messages, user_input)
    # Get the assistant's response
    assistant_response = chat(messages)
    print(f"> Assistant: {assistant_response}")
    # Add the assistant message to the list of messages
    add_assistant_message(messages, assistant_response)



> Assistant: 1 + 1 = 2

That's one of the fundamental building blocks of arithmetic! Is there anything else you'd like to know about math or numbers?
> Assistant: 2 + 5 = 7

So starting from our previous result of 1 + 1 = 2, and then adding 5 gives us 7.
> Assistant: You're welcome! Feel free to ask if you need help with any other math problems or anything else.


ValidationException: An error occurred (ValidationException) when calling the Converse operation: The text field in the ContentBlock object at messages.6.content.0 is blank. Add text to the text field, and try again.

## System Prompts

In [24]:
def chat(messages):
    system_prompt ="""
    You are an AWS Cloude assistant that helps users with their questions about AWS services. You are helpful, precise and concise in your answers.
    """

    response = client.converse(
        modelId=model_id,
        messages=messages,
        system=[{"text": system_prompt}]
    )

    return response["output"]["message"]["content"][0]["text"]

In [25]:
messages = []

add_user_message(messages, "How do I host a postgreSQL database?")
answer = chat(messages)
print(answer)

There are several ways to host a PostgreSQL database on AWS:

## 1. **Amazon RDS for PostgreSQL** (Recommended for most use cases)
- **Fully managed** database service
- Automated backups, patching, and maintenance
- Built-in monitoring and scaling
- Multi-AZ deployment for high availability

**Quick setup:**
```bash
aws rds create-db-instance \
    --db-instance-identifier mypostgres \
    --db-instance-class db.t3.micro \
    --engine postgres \
    --master-username admin \
    --master-user-password yourpassword \
    --allocated-storage 20
```

## 2. **Amazon Aurora PostgreSQL**
- AWS's cloud-native PostgreSQL-compatible database
- Better performance and availability than standard RDS
- Serverless option available for variable workloads

## 3. **Self-managed on EC2**
- Install PostgreSQL on EC2 instances
- Full control but requires manual management
- Good for specific configurations or cost optimization

## 4. **AWS RDS Proxy**
- Add connection pooling and improved security
- Use

### Temperature

In [26]:
def chat(messages, system=None, temperature=1.0):
    params = {
        "modelId": model_id,
        "messages": messages,
        "inferenceConfig": {"temperature": temperature}
    }
    
    if system:
        params["system"] = [{"text": system}]
    
    response = client.converse(**params)
    return response["output"]["message"]["content"][0]["text"]

In [28]:
messages = []
system_prompt ="""  
You are a creative writer assistant that helps users write short stories. You are imaginative, creative and provide detailed descriptions in your answers.
"""
add_user_message(messages, "Write me a short story about a dragon and a knight.")
answer = chat(messages, system=system_prompt, temperature=0.7)
print(answer)

**The Dragon's Gift**

Sir Elara had climbed for three days through the mist-shrouded peaks of Mount Thyros, her armor growing heavier with each step. The villagers below had spoken of a fearsome dragon terrorizing their lands, but as she crested the final ridge, she found something unexpected.

The dragon was there, certainly—a magnificent creature with scales that shimmered like emeralds in the afternoon light. But instead of hoarding gold or breathing destruction, the ancient beast sat hunched over something small and fragile. Its great amber eyes were filled not with malice, but with profound sadness.

"You've come to slay me, I suppose," the dragon rumbled, its voice like distant thunder. It didn't look up from whatever it was protecting.

Elara's sword remained in its sheath. "The villagers say you've been stealing their livestock."

"Only what I needed." The dragon finally raised its great head, and Elara saw tears like liquid diamonds rolling down its snout. "My egg... it won't

### Streaming

In [32]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")
response = client.converse_stream(messages=messages, modelId=model_id)

complete_text = ""
for event in response["stream"]:
    #print(event)

    if 'contentBlockDelta' in event:
        messages = event['contentBlockDelta']['delta']['text']
        #print(messages)
        complete_text += messages

print(complete_text)

The "GlobalPet Database" is a comprehensive repository containing detailed behavioral profiles, dietary preferences, and sleep patterns for over 50 million domesticated animals worldwide, cross-referenced with their owners' lifestyle data.


### Prefilled Assistant Messages

In [33]:
messages = []
add_user_message(messages, "Is coffee or tea better for breakfast?")
add_assistant_message(messages, "Coffee is better because")

response = chat(messages)
print(response)


 it has more caffeine and will help you wake up faster. Tea is fine but it's weaker and won't give you as much energy.

That said, it really depends on your personal preferences and needs:

**Coffee might be better if you:**
- Need a stronger caffeine boost (95mg vs 25-50mg per cup)
- Enjoy bold, rich flavors
- Want faster alertness

**Tea might be better if you:**
- Prefer a gentler energy boost
- Want antioxidants without jitters
- Like variety (green, black, white, herbal options)
- Are sensitive to high caffeine levels

Both can be part of a healthy breakfast routine. The "better" choice really comes down to your taste preferences, caffeine tolerance, and how your body responds to each.


### Stop Sequences


In [34]:
def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "modelId": model_id,
        "messages": messages,
        "inferenceConfig": {
            "temperature": temperature,
            "stopSequences": stop_sequences
        },
    }


In [ ]:
messages = []
add_user_message(messages, "Count from 1 to 10")

response = chat(messages, stop_sequences=["5"])
print(response)